# ⚽💰 Football Finance Intelligence
### Analyser les joueurs de football comme des actifs financiers

**Auteur :** Ahmed  Badi -  Ingénieur financier (EMSI Rabat)

## 🎯 Objectif
Appliquer les outils de l'analyse financière (croissance, décomposition des écarts, rentabilité, risque, valorisation) au marché des transferts du football, puis utiliser des agents IA pour interpréter les résultats.

## 📂 Source des données
Données **Transfermarkt** issues du projet open source *transfermarkt-datasets* : une base DuckDB de 12 tables (joueurs, clubs, transferts, valeurs marchandes, matchs…), mise à jour chaque semaine.

## 🧭 Module 1 — Le marché des transferts
1. Chargement des données
2. Contrôle qualité
3. Évolution du marché
4. Décomposition volume / prix
5. Focus sur les 5 grands championnats
6. Test du biais de survie
7. Concentration du marché (indice HHI)

In [1]:
!pip install duckdb -q
!curl -LO https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  201M  100  201M    0     0  11.3M      0  0:00:17  0:00:17 --:--:-- 14.7M


## 1. Chargement des données
On installe **DuckDB**, un moteur de base de données léger qui lit directement le fichier, sans serveur, et permet d'écrire des requêtes **SQL** très rapides. On télécharge ensuite la base Transfermarkt (environ 200 Mo).

In [2]:
import duckdb
con = duckdb.connect("transfermarkt-datasets.duckdb", read_only=True)

print(con.sql("SHOW TABLES"))
print(con.sql("SELECT * FROM transfers LIMIT 5").df())

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ appearances       │
│ club_games        │
│ clubs             │
│ competitions      │
│ countries         │
│ game_events       │
│ game_lineups      │
│ games             │
│ national_teams    │
│ player_valuations │
│ players           │
│ transfers         │
│ version           │
├───────────────────┤
│      13 rows      │
└───────────────────┘

   player_id transfer_date transfer_season  from_club_id  to_club_id  \
0     467994    2030-06-30           25/26          5621         749   
1     645842    2028-02-02           27/28          6505       19684   
2     677470    2028-02-02           27/28          6505        3535   
3     709155    2028-02-02           27/28          6505       19684   
4     728133    2028-02-02           27/28          6505       35759   

    from_club_name to_club_name  transfer_fee  market_value_in_eur  \
0         Reggiana    FC Empoli           0.0           

### Lecture
La base contient **13 tables** : les 12 tables de données et une table `version`. Celle qui nous intéresse ici est `transfers` : une ligne par transfert, avec le joueur, le club de départ, le club d'arrivée, le prix et la valeur marchande.

**Premières observations :**
- Certaines **dates sont dans le futur** (2028, 2030) : ce sont des transferts déjà programmés, comme des retours de prêt. Ils seront exclus.
- Beaucoup de transferts ont un **prix égal à 0** : prêts, retours de prêt ou joueurs libres. Ils n'ont pas de valeur financière pour notre analyse.

In [3]:
def q(sql):
    return con.sql(sql).df()

q("DESCRIBE transfers")

,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,transfer_date,DATE,YES,None,None,None
2,transfer_season,VARCHAR,YES,None,None,None
3,from_club_id,INTEGER,YES,None,None,None
4,to_club_id,INTEGER,YES,None,None,None
5,from_club_name,VARCHAR,YES,None,None,None
6,to_club_name,VARCHAR,YES,None,None,None
7,transfer_fee,"DECIMAL(18,3)",YES,None,None,None
8,market_value_in_eur,"DECIMAL(18,3)",YES,None,None,None
9,player_name,VARCHAR,YES,None,None,None


### Structure de la table `transfers`
| Colonne | Signification |
|---|---|
| `transfer_date` / `transfer_season` | Date et saison du transfert |
| `from_club_name` / `to_club_name` | Club vendeur / club acheteur |
| `transfer_fee` | **Prix payé** (en euros) |
| `market_value_in_eur` | Valeur marchande du joueur au moment du transfert |

La comparaison entre `transfer_fee` et `market_value_in_eur` permettra plus tard de mesurer si un club a **surpayé** ou fait une **bonne affaire**.

In [4]:
q("""
SELECT
    COUNT(*)                                          AS nb_total,
    MIN(transfer_date)                                AS date_min,
    MAX(transfer_date)                                AS date_max,
    SUM(CASE WHEN transfer_date > CURRENT_DATE THEN 1 ELSE 0 END) AS futurs,
    SUM(CASE WHEN transfer_fee IS NULL THEN 1 ELSE 0 END)         AS fee_manquant,
    SUM(CASE WHEN transfer_fee = 0 THEN 1 ELSE 0 END)             AS fee_zero,
    SUM(CASE WHEN transfer_fee > 0 THEN 1 ELSE 0 END)             AS fee_payant
FROM transfers
""")

,nb_total,date_min,date_max,futurs,fee_manquant,fee_zero,fee_payant
0,175165,1993-07-01,2030-06-30,475.0,61526.0,96085.0,17554.0


## 2. Contrôle qualité
| Catégorie | Nombre | Part |
|---|---|---|
| Total des transferts | 175 165 | 100 % |
| Prix inconnu | 61 526 | 35 % |
| Prix égal à 0 (prêts, joueurs libres) | 96 085 | 55 % |
| **Prix payant connu** | **17 554** | **10 %** |
| Dates futures | 475 | — |

**Interprétation :** seul **1 transfert sur 10** a un prix connu et positif. C'est cohérent avec la réalité du football, où la majorité des mouvements de joueurs sont des prêts ou des transferts gratuits. L'analyse financière portera sur ces **17 554 transferts payants**.

In [5]:
q("""
SELECT
    transfer_season,
    COUNT(*)                         AS nb_transferts_payants,
    ROUND(SUM(transfer_fee)/1e6, 1)  AS depenses_M_EUR,
    ROUND(AVG(transfer_fee)/1e6, 2)  AS fee_moyen_M_EUR
FROM transfers
WHERE transfer_fee > 0
  AND transfer_date <= CURRENT_DATE
GROUP BY transfer_season
ORDER BY MIN(transfer_date)
""")

,transfer_season,nb_transferts_payants,depenses_M_EUR,fee_moyen_M_EUR
0,97/98,1,0.0,0.00
1,02/03,3,1.0,0.33
2,03/04,1,19.0,19.00
3,04/05,12,17.4,1.45
4,05/06,16,39.8,2.49
5,06/07,26,59.6,2.29
6,07/08,60,211.9,3.53
7,08/09,69,167.5,2.43
8,09/10,95,355.4,3.74
9,10/11,143,449.6,3.14


### Lecture
Les premières saisons sont presque vides : **1 seul transfert payant en 97/98, 3 en 02/03**. Le marché existait bien à cette époque. C'est la **base de données** qui l'enregistre mal. Il faut donc choisir une année de départ où les données deviennent suffisamment nombreuses.

In [6]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

saisons = q("""
SELECT
    transfer_season AS saison,
    CASE WHEN CAST(LEFT(transfer_season, 2) AS INT) > 50
         THEN 1900 + CAST(LEFT(transfer_season, 2) AS INT)
         ELSE 2000 + CAST(LEFT(transfer_season, 2) AS INT)
    END                                       AS annee,
    COUNT(*)                                  AS nb_transferts,
    CAST(SUM(transfer_fee) / 1e6 AS DOUBLE)    AS depenses_M,
    CAST(MEDIAN(transfer_fee) / 1e6 AS DOUBLE) AS fee_median_M
FROM transfers
WHERE transfer_fee > 0
  AND transfer_date <= CURRENT_DATE
GROUP BY 1, 2
ORDER BY annee
""")

# Graphique de contrôle : combien de transferts par saison ?
fig = px.bar(saisons, x="saison", y="nb_transferts",
             title="Couverture des données : nombre de transferts payants par saison")
fig.show()

## 3. Évolution du marché
### Choix de la période d'analyse
Le nombre de transferts enregistrés devient significatif à partir de **2012/13** (environ 280 transferts). On retient donc la période **2012/13 – 2025/26**.

On exclut la saison **26/27**, qui est **en cours** : le mercato d'hiver n'a pas encore eu lieu.

⚠️ **Point d'attention :** le nombre de transferts augmente de façon continue (×8 en 13 ans). Une partie de cette hausse reflète probablement une **meilleure couverture des données**, et pas uniquement la croissance réelle du marché.

In [7]:
ANNEE_DEBUT = 2012   # ⬅️ remplace par l'année que tu as repérée
ANNEE_FIN   = 2025   # dernière saison complète (25/26)

df = saisons[(saisons["annee"] >= ANNEE_DEBUT) & (saisons["annee"] <= ANNEE_FIN)]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=df["saison"], y=df["depenses_M"],
                     name="Dépenses (M€)"), secondary_y=False)
fig.add_trace(go.Scatter(x=df["saison"], y=df["nb_transferts"],
                         name="Nombre de transferts", mode="lines+markers"),
              secondary_y=True)
fig.update_layout(title="Évolution du marché des transferts",
                  hovermode="x unified")
fig.update_yaxes(title_text="Dépenses (M€)", secondary_y=False)
fig.update_yaxes(title_text="Nombre de transferts", secondary_y=True)
fig.show()

# Calcul du TCAM
v_debut = df["depenses_M"].iloc[0]
v_fin   = df["depenses_M"].iloc[-1]
n       = len(df) - 1
tcam = (v_fin / v_debut) ** (1 / n) - 1
print(f"Dépenses {df['saison'].iloc[0]} : {v_debut:,.0f} M€")
print(f"Dépenses {df['saison'].iloc[-1]} : {v_fin:,.0f} M€")
print(f"TCAM sur {n} ans : {tcam:.1%} par an")

Dépenses 12/13 : 1,002 M€
Dépenses 25/26 : 12,102 M€
TCAM sur 13 ans : 21.1% par an


### Lecture du graphique
- **2012 → 2019 : croissance continue.** Les dépenses passent d'environ **1 Md€ à 7,7 Md€**.
- **2020/21 : choc du Covid (−37 %).** Stades vides et revenus en chute : les clubs ont subi un **choc de liquidité** et réduit leurs achats.
- **2021/22 : reprise lente**, puis forte accélération à partir de 2022/23.
- **2025/26 : record** à environ **12,1 Md€**.
- **2023/24 et 2024/25 :** les dépenses stagnent (environ 10,2 Md€) alors que le nombre de transferts augmente, donc le **prix moyen baisse**.

**TCAM (taux de croissance annuel moyen) : 21,1 % par an.**
⚠️ Ce chiffre est **surestimé** : il mélange la croissance réelle du marché et l'amélioration de la couverture des données. Les sections suivantes cherchent à séparer ces deux effets.

In [8]:
df = df.copy()
df["prix_moyen_M"] = df["depenses_M"] / df["nb_transferts"]

N0 = df["nb_transferts"].shift(1)
P0 = df["prix_moyen_M"].shift(1)
df["effet_volume"] = (df["nb_transferts"] - N0) * P0
df["effet_prix"]   = (df["prix_moyen_M"] - P0) * df["nb_transferts"]
df["variation"]    = df["depenses_M"].diff()
df["variation_%"]  = df["depenses_M"].pct_change() * 100

display(df[["saison", "nb_transferts", "prix_moyen_M", "depenses_M",
            "variation_%", "effet_volume", "effet_prix"]].round(1))

fig = go.Figure()
fig.add_trace(go.Bar(x=df["saison"], y=df["effet_volume"], name="Effet volume"))
fig.add_trace(go.Bar(x=df["saison"], y=df["effet_prix"],   name="Effet prix"))
fig.update_layout(barmode="relative",
                  title="D'où vient la variation des dépenses ? (M€)",
                  hovermode="x unified")
fig.show()

,saison,nb_transferts,prix_moyen_M,depenses_M,variation_%,effet_volume,effet_prix
11,12/13,280,3.6,1002.2,NaN,NaN,NaN
12,13/14,373,4.7,1771.7,76.8,332.9,436.6
13,14/15,460,4.3,1966.3,11.0,413.2,-218.6
14,15/16,620,4.6,2831.8,44.0,683.9,181.5
15,16/17,741,5.1,3757.4,32.7,552.7,372.9
16,17/18,936,5.8,5423.4,44.3,988.8,677.2
17,18/19,1115,5.1,5711.4,5.3,1037.2,-749.2
18,19/20,1315,5.8,7668.4,34.3,1024.5,932.5
19,20/21,1095,4.4,4807.7,-37.3,-1282.9,-1577.8
20,21/22,1343,3.9,5295.9,10.2,1088.9,-600.7


## 4. Décomposition volume / prix
**Principe :** dépenses = nombre de transferts × prix moyen. Une variation des dépenses peut donc venir :
- d'un **effet volume** : on achète plus ou moins de joueurs ;
- d'un **effet prix** : on achète des joueurs plus ou moins chers.

C'est la même méthode que l'**analyse des écarts** en contrôle de gestion.

### Lecture
| Saison | Constat | Interprétation |
|---|---|---|
| 2020/21 | Volume −1 283 M€, prix −1 578 M€ | Covid : moins d'achats **et** moins chers ; les grosses opérations ont été abandonnées en premier |
| 2018/19, 2024/25 | Volume ↑, prix ↓ | Saisons de **rationalisation** : plus de joueurs, mais moins chers |
| 2025/26 | Prix +1 446 M€, volume +374 M€ | Le record vient surtout de l'**inflation des prix** |

**Conclusion :** jusqu'en 2023, la croissance était surtout portée par le **volume**. En 2025/26, elle est portée par les **prix**, ce qui est le signe d'un marché qui se tend.

In [9]:
big5 = q("""
SELECT
    t.transfer_season AS saison,
    CASE WHEN CAST(LEFT(t.transfer_season, 2) AS INT) > 50
         THEN 1900 + CAST(LEFT(t.transfer_season, 2) AS INT)
         ELSE 2000 + CAST(LEFT(t.transfer_season, 2) AS INT) END AS annee,
    c.domestic_competition_id AS ligue,
    COUNT(*)                               AS nb_transferts,
    CAST(SUM(t.transfer_fee)/1e6 AS DOUBLE) AS depenses_M
FROM transfers t
JOIN clubs c ON t.to_club_id = c.club_id
WHERE t.transfer_fee > 0
  AND t.transfer_date <= CURRENT_DATE
  AND c.domestic_competition_id IN ('GB1','ES1','IT1','L1','FR1')
GROUP BY 1, 2, 3
ORDER BY annee
""")
big5 = big5[(big5["annee"] >= ANNEE_DEBUT) & (big5["annee"] <= ANNEE_FIN)]

noms = {"GB1": "Premier League", "ES1": "LaLiga", "IT1": "Serie A",
        "L1": "Bundesliga", "FR1": "Ligue 1"}
big5["ligue"] = big5["ligue"].map(noms)

fig = px.bar(big5, x="saison", y="depenses_M", color="ligue",
             title="Dépenses des clubs du Big 5 par ligue (M€)")
fig.show()

tot = big5.groupby("annee")["depenses_M"].sum()
n = len(tot) - 1
print(f"TCAM Big 5 : {(tot.iloc[-1] / tot.iloc[0]) ** (1/n) - 1:.1%} par an")

TCAM Big 5 : 21.0% par an


## 5. Focus sur les 5 grands championnats
Pour limiter l'effet de couverture, on se restreint aux clubs des 5 grands championnats, en principe mieux suivis.

### Lecture
- La **Premier League** représente environ **la moitié** des dépenses du Big 5 en 2025/26 (environ 4,1 Md€ sur 7,9 Md€).
- **LaLiga en 2019/20** : pic exceptionnel (Hazard au Real Madrid, Griezmann au Barça, João Félix à l'Atlético).

- La **Serie A** est généralement la **2ᵉ ligue la plus dépensière**. **LaLiga**, malgré son pic de 2019/20, dépense comparativement peu depuis 2020/21.

### ⚠️ Un résultat inattendu
Le TCAM du Big 5 reste à **21,0 %**, quasiment identique. Et 0,7 Md€ de dépenses pour tout le Big 5 en 2012/13 est **trop faible** par rapport à la réalité de l'époque.

**Hypothèse : un biais de survie.** La base est probablement construite à partir des joueurs suivis aujourd'hui. Les joueurs transférés il y a longtemps et retraités depuis seraient donc sous-représentés. Cette hypothèse est testée dans la section suivante.

*Limite : la table `clubs` indique le championnat **actuel** du club. Un club relégué peut donc sortir de l'échantillon.*

In [11]:
q("DESCRIBE players")

,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,first_name,VARCHAR,YES,None,None,None
2,last_name,VARCHAR,YES,None,None,None
3,name,VARCHAR,YES,None,None,None
4,last_season,VARCHAR,YES,None,None,None
5,current_club_id,VARCHAR,YES,None,None,None
6,player_code,VARCHAR,YES,None,None,None
7,country_of_birth,VARCHAR,YES,None,None,None
8,city_of_birth,VARCHAR,YES,None,None,None
9,country_of_citizenship,VARCHAR,YES,None,None,None


In [12]:
q("""
SELECT
    t.transfer_season AS saison,
    COUNT(*) AS nb,
    ROUND(100.0 * AVG(
        CASE WHEN COALESCE(TRY_CAST(p.last_season AS INT),
                           2000 + TRY_CAST(LEFT(p.last_season, 2) AS INT)) >= 2024
             THEN 1 ELSE 0 END
    ), 1) AS pct_encore_actifs
FROM transfers t
JOIN players p ON t.player_id = p.player_id
WHERE t.transfer_fee > 0
  AND t.transfer_date <= CURRENT_DATE
GROUP BY 1
ORDER BY MIN(t.transfer_date)
""")

,saison,nb,pct_encore_actifs
0,97/98,1,100.0
1,02/03,3,100.0
2,03/04,1,100.0
3,04/05,12,75.0
4,05/06,16,81.3
5,06/07,26,76.9
6,07/08,60,66.7
7,08/09,69,72.5
8,09/10,95,80.0
9,10/11,143,81.1


### Résultat : le biais de survie est confirmé
En 2012/13, **85 %** des joueurs transférés sont encore actifs en 2024, soit 12 ans plus tard. En 2007/08, ils sont encore **67 %** après 17 ans. Dans la réalité, la grande majorité de ces joueurs ont pris leur retraite.

**Conclusion :** la base contient surtout les transferts des joueurs **encore suivis aujourd'hui**. Plus une saison est ancienne, plus ses transferts sont sous-représentés. C'est ce qui gonfle artificiellement la croissance mesurée (TCAM de 21 %).

*(Les saisons avant 2004 ne comptent que 1 à 3 transferts : trop peu pour conclure.)*

### Conséquences pour la suite de l'analyse
- Les **montants en euros** doivent être lus comme les « dépenses **observées dans la base** », et non comme le total réel du marché.
- On privilégie les indicateurs **relatifs**, moins sensibles à ce biais : **parts de marché**, **prix médian**, **ratios**.
- Les comparaisons dans le temps sont plus fiables sur les **saisons récentes**.

## 7. Concentration du marché (indice HHI)
**Question :** l'argent est-il partagé entre les ligues, ou concentré dans une seule ?

**Méthode :** l'indice **HHI** additionne les carrés des parts de marché (en %).
- 5 ligues égales (20 % chacune) → **HHI = 2 000**
- Une seule ligue qui fait tout → **HHI = 10 000**

Les parts de marché comparent les ligues entre elles : elles sont donc **beaucoup moins sensibles** au biais de survie.

In [13]:
parts = big5.pivot_table(index="saison", columns="ligue",
                         values="depenses_M", aggfunc="sum", sort=False)
parts = parts.div(parts.sum(axis=1), axis=0) * 100

fig = px.area(parts, title="Parts de marché des dépenses au sein du Big 5 (%)")
fig.update_layout(hovermode="x unified", yaxis_title="%")
fig.show()

hhi = (parts ** 2).sum(axis=1)
fig = px.line(x=hhi.index, y=hhi.values, markers=True,
              title="Concentration du marché (indice HHI)")
fig.add_hline(y=2000, line_dash="dash", annotation_text="Répartition égale (2000)")
fig.update_layout(xaxis_title="saison", yaxis_title="HHI")
fig.show()

print(parts.round(1).tail(3))

ligue   Premier League  LaLiga  Ligue 1  Serie A  Bundesliga
saison                                                      
23/24             45.4     8.6     17.4     15.9        12.7
24/25             44.8     9.3     14.5     19.0        12.5
25/26             52.2     9.6      9.3     17.1        11.8


### Lecture
**1. Une Premier League dominante, et de plus en plus**
Sa part passe de **45 %** (2023/24 et 2024/25) à **52 %** en 2025/26 : elle dépense à elle seule plus que les 4 autres ligues réunies. Son point bas est en 2019/20 (environ 29 %), lors de l'été record de LaLiga.

**2. Un classement instable derrière elle**
| Ligue | 2023/24 | 2024/25 | 2025/26 |
|---|---|---|---|
| Premier League | 45,4 % | 44,8 % | **52,2 %** |
| Serie A | 15,9 % | **19,0 %** | 17,1 % |
| Bundesliga | 12,7 % | 12,5 % | 11,8 % |
| Ligue 1 | **17,4 %** | 14,5 % | 9,3 % |
| LaLiga | 8,6 % | 9,3 % | 9,6 % |

- La **Ligue 1** voit sa part presque divisée par deux en 2 ans (17,4 % → 9,3 %).
- **LaLiga** est la ligue la moins dépensière du Big 5 sur les 3 dernières saisons.

**3. Concentration (HHI)**
- Le HHI reste **toujours au-dessus de 2 000** : le marché n'a jamais été équilibré.
- Minimum en **2019/20** (environ 2 180), maximum en **2022/23** (environ 3 430).
- **Depuis le Covid, il ne redescend plus sous 2 700** : la concentration est devenue durable.

**Nombre de ligues équivalentes (10 000 ÷ HHI) :** environ **4,6** en 2019/20 contre environ **3,0** en 2025/26. Le marché fonctionne désormais comme s'il ne comptait que 3 ligues de poids égal.

*Ces résultats reposent sur des parts de marché, donc ils sont peu sensibles au biais de survie identifié plus haut.*

## 8. Les flux d'argent entre ligues (2023/24 – 2025/26)
**Question :** quelles ligues **achètent** des talents, et lesquelles les **vendent** ?

**Méthode :** un **diagramme de Sankey** relie la ligue du club vendeur (à gauche) à celle du club acheteur (à droite). L'épaisseur de chaque flèche est proportionnelle au montant. On calcule ensuite le **solde net** de chaque ligue : ventes − achats.
- Solde **positif** → ligue **exportatrice** de talents (elle encaisse)
- Solde **négatif** → ligue **importatrice** (elle dépense)

On se limite aux 3 dernières saisons complètes, les plus fiables de la base.

In [15]:
import pandas as pd

flux = q("""
SELECT
    CASE WHEN cf.domestic_competition_id IN ('GB1','ES1','IT1','L1','FR1')
         THEN cf.domestic_competition_id ELSE 'AUTRES' END AS depart,
    CASE WHEN ct.domestic_competition_id IN ('GB1','ES1','IT1','L1','FR1')
         THEN ct.domestic_competition_id ELSE 'AUTRES' END AS arrivee,
    CAST(SUM(t.transfer_fee) / 1e6 AS DOUBLE) AS montant_M
FROM transfers t
LEFT JOIN clubs cf ON t.from_club_id = cf.club_id
LEFT JOIN clubs ct ON t.to_club_id   = ct.club_id
WHERE t.transfer_fee > 0
  AND t.transfer_date <= CURRENT_DATE
  AND t.transfer_season IN ('23/24', '24/25', '25/26')
GROUP BY 1, 2
""")

noms2 = {**noms, "AUTRES": "Autres ligues"}
cles  = list(noms2.keys())
labels = [noms2[k] + " (vend)" for k in cles] + [noms2[k] + " (achète)" for k in cles]

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=15, thickness=18),
    link=dict(
        source=[cles.index(d) for d in flux["depart"]],
        target=[len(cles) + cles.index(a) for a in flux["arrivee"]],
        value=flux["montant_M"].tolist(),
    ),
))
fig.update_layout(title="Flux d'argent entre ligues, 2023/24 – 2025/26 (M€)", height=600)
fig.show()

achats = flux.groupby("arrivee")["montant_M"].sum()
ventes = flux.groupby("depart")["montant_M"].sum()
bilan = pd.DataFrame({"ventes_M": ventes, "achats_M": achats}).fillna(0)
bilan["solde_net_M"] = bilan["ventes_M"] - bilan["achats_M"]
bilan.index = bilan.index.map(noms2)
bilan.sort_values("solde_net_M").round(0)

,ventes_M,achats_M,solde_net_M
Premier League,7043.0,10237.0,-3194.0
Serie A,3472.0,3710.0,-239.0
LaLiga,2332.0,1967.0,365.0
Bundesliga,3066.0,2641.0,425.0
Ligue 1,3405.0,2889.0,516.0
Autres ligues,13319.0,11191.0,2128.0


### Lecture
| Ligue | Ventes | Achats | Solde net | Profil |
|---|---|---|---|---|
| Premier League | 7 043 M€ | 10 237 M€ | **−3 194 M€** | Importatrice nette |
| Serie A | 3 472 M€ | 3 710 M€ | −239 M€ | Quasi à l'équilibre |
| LaLiga | 2 332 M€ | 1 967 M€ | +365 M€ | Légèrement exportatrice |
| Bundesliga | 3 066 M€ | 2 641 M€ | +425 M€ | Légèrement exportatrice |
| Ligue 1 | 3 405 M€ | 2 889 M€ | +516 M€ | Légèrement exportatrice |
| Autres ligues | 13 319 M€ | 11 191 M€ | **+2 128 M€** | Exportatrices nettes |

- La **Premier League** est la **seule ligue importatrice nette** : −3,2 Md€ sur 3 saisons. Elle est aussi le premier vendeur du Big 5, car une grande partie de son argent circule entre clubs anglais.
- Les **autres ligues** (championnats formateurs) sont les grandes exportatrices : elles forment des joueurs et les vendent aux grands championnats.

**✅ Contrôle de cohérence :** la somme des soldes nets est égale à **0**. Chaque euro payé par un acheteur est encaissé par un vendeur, comme une balance comptable équilibrée. Les calculs sont donc cohérents.

In [16]:
couleurs = {
    "GB1": (99, 110, 250),   # Premier League - bleu
    "ES1": (239, 85, 59),    # LaLiga - rouge
    "FR1": (0, 204, 150),    # Ligue 1 - vert
    "IT1": (171, 99, 250),   # Serie A - violet
    "L1":  (255, 161, 90),   # Bundesliga - orange
    "AUTRES": (150, 150, 150),
}
def rgba(cle, alpha):
    r, g, b = couleurs[cle]
    return f"rgba({r},{g},{b},{alpha})"

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=15, thickness=18,
              color=[rgba(k, 1) for k in cles] * 2),
    link=dict(
        source=[cles.index(d) for d in flux["depart"]],
        target=[len(cles) + cles.index(a) for a in flux["arrivee"]],
        value=flux["montant_M"].tolist(),
        color=[rgba(d, 0.4) for d in flux["depart"]],
        hovertemplate="%{source.label} → %{target.label}<br>%{value:,.0f} M€<extra></extra>",
    ),
))
fig.update_layout(title="Flux d'argent entre ligues, 2023/24 – 2025/26 (M€)", height=650)
fig.show()

# Part des transferts internes (une ligue qui achète à elle-même)
for k in cles:
    total_achats = flux.loc[flux["arrivee"] == k, "montant_M"].sum()
    interne = flux.loc[(flux["arrivee"] == k) & (flux["depart"] == k), "montant_M"].sum()
    print(f"{noms2[k]:<16} achats internes : {interne / total_achats:.0%}")

Premier League   achats internes : 40%
LaLiga           achats internes : 28%
Serie A          achats internes : 38%
Bundesliga       achats internes : 22%
Ligue 1          achats internes : 25%
Autres ligues    achats internes : 66%


### Lecture du Sankey
- **Toutes les ligues du Big 5 vendent à la Premier League** : les flux orange (Bundesliga), vert (Ligue 1), rouge (LaLiga) et violet (Serie A) convergent vers elle. C'est le **client final** du marché des transferts.
- **La Premier League achète en grande partie à elle-même** : une part importante de l'argent anglais circule entre clubs anglais.
- **Les « autres ligues » (championnats formateurs) approvisionnent tout le marché**, en particulier la Premier League et la Serie A.
- **La Premier League revend peu aux autres grands championnats** : le flux de talents est essentiellement **à sens unique**, vers l'Angleterre.

---
# ✅ Conclusion du Module 1 — Le marché des transferts

### Les 5 messages clés
1. **Un marché en forte croissance.** Les dépenses observées passent d'environ 1 Md€ (2012/13) à **12,1 Md€ (2025/26)**, un record.
2. **Le Covid, un choc de liquidité.** En 2020/21, les dépenses ont chuté de **−37 %** : les clubs ont acheté moins de joueurs, et surtout moins chers. La reprise devient forte à partir de 2022/23.
3. **Une croissance désormais tirée par les prix.** En 2025/26, la hausse vient surtout de l'**effet prix** (+1,4 Md€) plutôt que de l'effet volume (+0,4 Md€). Le marché se tend.
4. **Une concentration croissante.** La Premier League capte **52 %** des dépenses du Big 5. L'indice HHI montre que le marché fonctionne comme s'il ne comptait plus qu'environ **3 ligues** de poids égal (contre 4,6 en 2019/20).
5. **Des flux à sens unique.** La Premier League est la **seule importatrice nette** (−3,2 Md€ sur 3 saisons). Les championnats formateurs sont les grands exportateurs (+2,1 Md€).

### Limites
- **Biais de survie :** la base enregistre surtout les transferts des joueurs encore suivis aujourd'hui. Les montants anciens sont sous-estimés, donc la croissance réelle est **inférieure** au TCAM mesuré de 21 %.
- Les championnats sont attribués selon la **ligue actuelle** des clubs.
- Seuls **10 %** des transferts ont un prix connu et positif.

➡️ **Module suivant :** le joueur vu comme un actif financier (valeur, rendement, risque, amortissement).